In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Add the `src` directory to sys.path for local imports if the script is run directly
import sys
import numpy as np
if not __package__:
    sys.path.insert(0, '/home/mike/git/QDMpy/src')

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from QDMpy.odmr.data import ODMRData
from QDMpy.odmr.io import MatlabLoader
from QDMpy.odmr.processors import BinningProcessor
from QDMpy.odmr.odmr import ODMR
from QDMpy.fit import Fit


In [ ]:

# User-friendly initialization
loader = MatlabLoader(data_folder="/home/mike/git/QDMpy/tests/data")
odmr_data = ODMRData.from_loader(loader=loader)
odmr = ODMR(odmr_data)
odmr.processor_manager.add_processor(BinningProcessor(bin_factor=2))
odmr.process_data()

In [ ]:
from QDMpy.fit import Fit
from QDMpy import guess

In [ ]:
from QDMpy.models import ModelRegistry

In [ ]:
n_peaks, doubt, _ = guess.guess_n_peaks(odmr.processed_data.data)

In [ ]:
guess.get_model_by_peaks(n_peaks)

In [ ]:
from scipy.signal import find_peaks

In [ ]:
median_data = np.median(odmr.processed_data.data, axis=3)

In [ ]:
find_peaks(-median_data[0,0], prominence=0.000004)

In [ ]:
plt.plot(median_data[0,0])
plt.show()
# print(fit)

In [ ]:
import numpy as np
np.median(odmr.processed_data.data, axis=3).shape

In [ ]:
models.guess_model_name(odmr.processed_data.data)

In [ ]:
fit = Fit(odmr)

In [ ]:
models.IMPLEMENTED[fit.model_name]

In [ ]:
models.

In [ ]:
from __future__ import annotations

In [ ]:
guess_initial_fit_parameters(odmr.processed_data.data, odmr.processed_data.frequencies, models.IMPLEMENTED[fit.model_name]).shape

In [ ]:
from QDMpy.initial_guess import *

In [ ]:
def guess_initial_fit_parameters(
    data: NDArray, freq: NDArray, model: Dict[str, Any]
) -> NDArray:
    """
    Guess initial fit parameters based on the selected model.

    :param data: NDArray
        3D array of the data to fit (e.g., ODMR data).
    :param freq: NDArray
        1D array of the frequencies corresponding to the data.
    :param model: Dict[str, Any]
        Model dictionary containing parameter names and number of peaks.

    :return: NDArray
        Initial fit parameters as a 4D array of shape (n_pol, n_frange, n_pixel, n_params).
    """
    # Define parameter guessers for each parameter type
    parameter_guessers = {
        "center": lambda: guess_center(data, freq),
        "contrast": lambda: guess_contrast(data),
        "width": lambda: guess_width(
            data, freq, vmin=0.3, vmax=0.7
        ),  # Default vmin/vmax can be adjusted dynamically
        "offset": lambda: np.ones(data.shape[:-1]),  # Offset is often assumed to be 1
    }

    # Initialize list for parameter arrays
    fit_parameters = []

    # Guess each parameter defined in the model
    for param in model["params"]:
        if param in parameter_guessers:
            fit_parameters.append(parameter_guessers[param]())
        else:
            raise ValueError(f"Parameter {param} has no defined guess method.")

    # Stack parameters along the last axis
    return np.stack(fit_parameters, axis=-1)
